# RAG retriever evaluation

Запуск использует `config.py` и пакет `rag_eval`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_PATH = PROJECT_ROOT / "config.py"
CONFIG_PATH

PosixPath('/home/vladislav/Рабочий стол/working/test_system/config.py')

Перед боевым запуском проверьте в `config.py`: `PIPELINE_FACTORY`, `RETRIEVER_INIT_KWARGS`, `RETRIEVER_TOP_K`, `RAGAS_JUDGE_PROVIDER`, параметры `QWEN_*` или `GIGACHAT_*`.

In [2]:
from rag_eval import EvaluationRunner, MetricsCalculator

run_file = EvaluationRunner.from_config(CONFIG_PATH).run()
metrics_file = MetricsCalculator.from_config(CONFIG_PATH).evaluate(run_file)

{"run_file": run_file, "metrics_file": metrics_file}

Инициализация: CPU | Fusion: RRF | α=0.5
✅ Загружено 10 документов


Running RAG:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Running custom judge:   0%|          | 0/10 [00:00<?, ?it/s]

/home/vladislav/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

{'run_file': PosixPath('/home/vladislav/Рабочий стол/working/test_system/outputs/runs/rag_run_20260630_235539_674876.xlsx'),
 'metrics_file': PosixPath('/home/vladislav/Рабочий стол/working/test_system/outputs/metrics/metrics_summary.xlsx')}

## Grid search параметров retriever

Перебор выключен по умолчанию. Укажите сетку, целевую метрику и поставьте `RUN_GRID_SEARCH = True`. В перебор попадают только параметры, присутствующие в `RETRIEVER_INIT_KWARGS`, поэтому `rerank_initial_k` автоматически пропускается для retriever-only. Каждый trial создаёт отдельный run со снимком параметров.

Если run-файл уже создан и нужно только пересчитать метрики, укажите путь ниже.

In [2]:
from rag_eval import MetricsCalculator
metrics_file = MetricsCalculator.from_config(CONFIG_PATH).evaluate(
    "outputs/runs/rag_run_20260630_235539_674876.xlsx"
)
metrics_file

Calculating metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Running custom judge:   0%|          | 0/10 [00:00<?, ?it/s]

/home/vladislav/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

PosixPath('/home/vladislav/Рабочий стол/working/test_system/outputs/metrics/metrics_summary.xlsx')

In [ ]:
from copy import deepcopy
from itertools import product
import json
import pandas as pd

from rag_eval import AppConfig, EvaluationRunner, MetricsCalculator

RUN_GRID_SEARCH = False
OBJECTIVE_METRIC = "retriever_ndcg_at_10_mean"
PARAM_GRID = {
    "fusion_method": ["rrf"],
    "k_rrf": [30, 60, 90],
    "alpha": [0.3, 0.5, 0.7],
    "bias": [0.0],
    "rerank_initial_k": [30, 50],
}

grid_results = pd.DataFrame()
if RUN_GRID_SEARCH:
    base_config = AppConfig.from_file(CONFIG_PATH)
    active_grid = {
        name: values
        for name, values in PARAM_GRID.items()
        if name in base_config.retriever_adapter.init_kwargs
    }
    skipped = sorted(set(PARAM_GRID) - set(active_grid))
    if skipped:
        print("Пропущены параметры, которых нет в RETRIEVER_INIT_KWARGS:", skipped)

    records = []
    names = list(active_grid)
    for values in product(*(active_grid[name] for name in names)):
        trial_params = dict(zip(names, values))
        trial_config = deepcopy(base_config)
        trial_config.retriever_adapter.init_kwargs.update(trial_params)
        run_path = EvaluationRunner(trial_config).run()
        metrics_path = MetricsCalculator(trial_config).evaluate(run_path)
        metrics_json = json.loads(metrics_path.with_suffix(".json").read_text(encoding="utf-8"))
        summary = metrics_json["summary"][-1]
        records.append({**trial_params, **summary})

    grid_results = pd.DataFrame(records)
    if OBJECTIVE_METRIC in grid_results:
        grid_results = grid_results.sort_values(OBJECTIVE_METRIC, ascending=False).reset_index(drop=True)
else:
    print("Grid search выключен: установите RUN_GRID_SEARCH = True")

grid_results